In [1]:
%reload_ext autoreload
%autoreload 2

import os
from pathlib import Path

print(Path().cwd())
os.chdir(Path(os.getcwd()).parent)
print(Path().cwd())

/Users/samantha/QuantUS-Plugins-CEUS/China_Data
/Users/samantha/QuantUS-Plugins-CEUS


## Select Contrast-Enhanced Ultrasound (CEUS) Cine and Parser

In [2]:
from src.image_loading.options import get_scan_loaders

print("Available scan loaders:", list(get_scan_loaders().keys()))

Available scan loaders: ['avi', 'nifti', 'custom_dicom', 'mp4']


In [67]:
scan_type = 'nifti'

scan_path = '/Users/samantha/Desktop/tul/china data/p34/new_v2/CEUS-26285-2.nii.gz'
scan_loader_kwargs = {
    'transpose': False,
}

In [68]:
from src.entrypoints import scan_loading_step

image_data = scan_loading_step(scan_type, scan_path, **scan_loader_kwargs)

## Load Segmentation

Assumes same segmentation for each frame

In [69]:
from src.seg_loading.options import get_seg_loaders

print("Available segmentation loaders:", list(get_seg_loaders().keys()))

Available segmentation loaders: ['nifti', 'load_bolus_mask']


In [98]:
seg_type = 'nifti'

seg_path = '/Users/samantha/Desktop/tul/china data/p34/new_v2/manual_vois/v2_necrosis_removed.nii.gz'
seg_loader_kwargs = {}

In [99]:
from src.entrypoints import seg_loading_step

seg_data = seg_loading_step(seg_type, image_data, seg_path, scan_path, **seg_loader_kwargs)

## CEUS Quantitative Temporal Curve Analysis

In [100]:
from src.time_series_analysis.options import get_analysis_types, get_required_kwargs

all_analysis_types, all_analysis_funcs = get_analysis_types()
print("Available analysis types:", list(all_analysis_types.keys()))

Available analysis types: ['curves_paramap', 'curves']


In [101]:
analysis_type = 'curves'

print("Available analysis functions:", list(all_analysis_funcs.keys()))

Available analysis functions: ['pyradiomics', 'tic']


In [102]:
analysis_funcs = ['tic']

# Find all required kwargs for the analysis functions
analysis_funcs = analysis_funcs if len(analysis_funcs) else list(all_analysis_funcs[analysis_type].keys())
required_kwargs = get_required_kwargs(analysis_type, analysis_funcs)
print("Required kwargs for current analysis:", required_kwargs)

Required kwargs for current analysis: []


In [103]:
# Set frame rate (adjust this value to match your actual video fps)
image_data.frame_rate = 1  # e.g., 30 fps

analysis_kwargs = {
    'ax_vox_ovrlp': 50,
    'sag_vox_ovrlp': 50,
    'cor_vox_ovrlp': 50,
    'ax_vox_len': 20.0,
    'sag_vox_len': 20.0,
    'cor_vox_len': 20.0,
}


In [104]:
from src.entrypoints import analysis_step

analysis_obj = analysis_step(analysis_type, image_data, seg_data, analysis_funcs, **analysis_kwargs)

Computing curves: 100%|██████████| 210/210 [00:03<00:00, 68.20it/s]


In [105]:
import numpy as np

# Get raw image and mask
img = image_data.intensities_for_analysis
mask = seg_data.seg_mask

# Recompute TIC to match GUI
tic = []
for t in range(img.shape[3]):
    frame = img[:, :, :, t]
    tic.append(np.exp(frame[mask > 0] / 24.09).mean())
tic = np.array(tic)

# Baseline subtraction (same as GUI)
tic = tic - np.mean(tic[:2])
if np.any(tic < 0):
    tic = tic + np.abs(np.min(tic))
else:
    tic = tic - np.min(tic)

# Replace the TIC in the analysis object
analysis_obj.curves[0]['TIC'] = list(tic)

## Curve Quantification

In [106]:
from src.curve_quantification.options import get_quantification_funcs

quantification_funcs = get_quantification_funcs()
print("Available quantification functions:", quantification_funcs.keys())

Available quantification functions: dict_keys(['auc_no_fit', 'cmus_firstorder', 'dte', 'first_order_full', 'first_order_select', 'lognormal_fit_full', 'lognormal_fit_select', 'wash_rates'])


In [107]:
function_names = [] # Empty list will use all functions
output_path = '/Users/samantha/Desktop/tul/china data/output/curve_quant_raw.csv'
curve_quantifications_kwargs = {
    'curves_to_fit': ['TIC'],
    'tic_name': 'TIC'
}

In [108]:
from src.entrypoints import curve_quantification_step

curve_quant = curve_quantification_step(analysis_obj, function_names, output_path, **curve_quantifications_kwargs)

In [109]:
import numpy as np
data = curve_quant.data_dict[0]
print(f"{'Parameter':<30} {'Value':>15}")
print("-" * 47)
for key, value in data.items():
    if isinstance(value, float):
        print(f"{key:<30} {value:>15.4f}")
    else:
        print(f"{key:<30} {str(value):>15}")

# Volume calculation
pixdim = image_data.pixdim
voxel_vol_mm3 = np.prod(pixdim)
n_voxels = int(np.sum(seg_data.seg_mask > 0))
vol_mm3 = n_voxels * voxel_vol_mm3

print(f"\n{'VOI Volume':<30}")
print("-" * 47)
print(f"{'Voxel spacing (mm)':<30} {str(pixdim):>15}")
print(f"{'Voxels in VOI':<30} {n_voxels:>15,}")
print(f"{'Volume (mm³)':<30} {vol_mm3:>15.1f}")
print(f"{'Volume (cm³)':<30} {vol_mm3/1000:>15.2f}")

Parameter                                Value
-----------------------------------------------
Scan Name                      CEUS-26285-2.nii
Segmentation Name              v2_necrosis_removed
AUC_NoFit_TIC                          49.5657
AUC_full_TIC                        37984.7709
PE_full_TIC                           690.3641
TP_full_TIC                            15.5303
MTT_full_TIC                           55.7482
T0_full_TIC                            11.5933
Mu_full_TIC                             3.5948
Sigma_full_TIC                          0.9231
PE_Ix_full_TIC                              27
WashIn_Mean_TIC                       238.1668
WashIn_Std_TIC                        283.3804
WashIn_Max_TIC                        732.4090
WashIn_Min_TIC                          0.0000
WashIn_Median_TIC                      33.2081
WashIn_Variance_TIC                 80304.4522
WashIn_Skewness_TIC                     0.6242
WashIn_Kurtosis_TIC                    -1.3295
WashIn_

In [110]:
import numpy as np
data = curve_quant.data_dict[0]
print(f"{'Parameter':<30} {'Value':>15}")
print("-" * 47)
for key, value in data.items():
    if isinstance(value, float):
        print(f"{key:<30} {value:>15.4f}")
    else:
        print(f"{key:<30} {str(value):>15}")

# Volume calculation
pixdim = image_data.pixdim
voxel_vol_mm3 = np.prod(pixdim)
n_voxels = int(np.sum(seg_data.seg_mask > 0))
vol_mm3 = n_voxels * voxel_vol_mm3

print(f"\n{'VOI Volume':<30}")
print("-" * 47)
print(f"{'Voxel spacing (mm)':<30} {str(pixdim):>15}")
print(f"{'Voxels in VOI':<30} {n_voxels:>15,}")
print(f"{'Volume (mm³)':<30} {vol_mm3:>15.1f}")
print(f"{'Volume (cm³)':<30} {vol_mm3/1000:>15.2f}")

Parameter                                Value
-----------------------------------------------
Scan Name                      CEUS-26285-2.nii
Segmentation Name              v2_necrosis_removed
AUC_NoFit_TIC                          49.5657
AUC_full_TIC                        37984.7709
PE_full_TIC                           690.3641
TP_full_TIC                            15.5303
MTT_full_TIC                           55.7482
T0_full_TIC                            11.5933
Mu_full_TIC                             3.5948
Sigma_full_TIC                          0.9231
PE_Ix_full_TIC                              27
WashIn_Mean_TIC                       238.1668
WashIn_Std_TIC                        283.3804
WashIn_Max_TIC                        732.4090
WashIn_Min_TIC                          0.0000
WashIn_Median_TIC                      33.2081
WashIn_Variance_TIC                 80304.4522
WashIn_Skewness_TIC                     0.6242
WashIn_Kurtosis_TIC                    -1.3295
WashIn_

## Save Results to CSV

In [111]:
import pandas as pd
import os
from datetime import datetime

out_path = "/Users/samantha/Desktop/tul/china data/p34/new_v2/manual_vois/paramap/necrosis_removed_voi.csv"

data = curve_quant.data_dict[0]
df = pd.DataFrame([data])
df["Timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

scan_col = "Scan Name"

if os.path.exists(out_path):
    existing = pd.read_csv(out_path)
    if "Timestamp" not in existing.columns:
        existing["Timestamp"] = pd.NaT
        existing.to_csv(out_path, index=False)
    df.to_csv(out_path, mode="a", header=False, index=False)
    print(f"Appended to {out_path}")
else:
    df.to_csv(out_path, index=False)
    print(f"Created {out_path}")

Created /Users/samantha/Desktop/tul/china data/p34/new_v2/manual_vois/paramap/necrosis_removed_voi.csv
